In [1]:
import pandas as pd
import numpy as np

from faker import Faker
import random
import os

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

In [2]:
# ── CONFIGURAÇÕES ──────────────────────────────────────────

REGIONS    = ['Sul', 'Sudeste', 'Centro-Oeste', 'Norte', 'Nordeste']
CATEGORIES = ['Notebooks', 'Periféricos', 'Smartphones', 'Acessórios', 'Monitores']

PRODUCTS = {
    'Notebooks':    ['Dell Inspiron', 'Lenovo IdeaPad', 'HP Pavilion', 'Asus VivoBook'],
    'Periféricos':  ['Mouse Logitech', 'Teclado Redragon', 'Headset HyperX', 'Webcam Logitech'],
    'Smartphones':  ['Samsung A54', 'Xiaomi 12', 'iPhone 13', 'Motorola Edge'],
    'Acessórios':   ['Cabo USB-C', 'Carregador 65W', 'Suporte Notebook', 'Hub USB'],
    'Monitores':    ['LG 24"', 'Dell 27"', 'Samsung 32"', 'AOC 24"'],
}

BASE_PRICE = {
    'Notebooks':    2800,
    'Periféricos':  180,
    'Smartphones':  1900,
    'Acessórios':   90,
    'Monitores':    1100,
}

COST_RATIO = {
    'Notebooks':    0.72,
    'Periféricos':  0.80,  # margem baixa — intencional
    'Smartphones':  0.74,
    'Acessórios':   0.65,
    'Monitores':    0.70,
}

# Quantidade máxima por categoria — realista para B2B
MAX_QTY = {
    'Notebooks':    2,
    'Periféricos':  10,
    'Smartphones':  3,
    'Acessórios':   20,
    'Monitores':    3,
}

# Sazonalidade das vendas
SEASONALITY = {
    1:  1.10,  # volta às aulas
    2:  0.85,
    3:  0.90,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.05,  # férias
    8:  0.95,
    9:  0.95,
    10: 1.00,
    11: 1.35,  # Black Friday
    12: 1.20,  # Natal
}

# Metas base 2022 por região
GOALS_2022 = {
    'Sul':          900000,
    'Sudeste':      1300000,
    'Centro-Oeste': 750000,
    'Norte':        600000,   # underperforma — intencional
    'Nordeste':     950000,
}

# Fator de sazonalidade nas metas
GOAL_SEASONALITY = {
    1:  1.05,
    2:  0.90,
    3:  0.95,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.00,
    8:  0.95,
    9:  1.00,
    10: 1.05,
    11: 1.25,  # Black Friday
    12: 1.15,  # Natal
}

# Crescimento anual das metas
GROWTH_2023 = 1.12  # 12% de crescimento

In [3]:
# ── DIM_SELLER ─────────────────────────────────────────────

sellers = []
seller_id = 1

for region in REGIONS:
    n_sellers = 6 if region == 'Sudeste' else 4
    for _ in range(n_sellers):
        seniority = round(random.uniform(0.5, 8.0), 1)
        sellers.append({
            'vendedor_id':       seller_id,
            'nome_vendedor':     fake.name(),
            'regiao':            region,
            'anos_experiencia':  seniority,
            'equipe':            'Comercial',
        })
        seller_id += 1

dim_seller = pd.DataFrame(sellers)


In [4]:
# ── DIM_METAS ──────────────────────────────────────────────

metas = []
meta_id = 1

for mes in pd.date_range(start='2022-01-01', end='2023-12-31', freq='MS'):
    year  = mes.year
    month = mes.month

    for regiao, meta_base in GOALS_2022.items():
        # Crescimento em 2023
        meta_ano = meta_base if year == 2022 else meta_base * GROWTH_2023

        # Sazonalidade na meta
        meta_final = round(meta_ano * GOAL_SEASONALITY[month], 2)

        metas.append({
            'meta_id':     meta_id,
            'regiao':      regiao,
            'mes':         mes.strftime('%Y-%m'),
            'meta_mensal': meta_final,
        })
        meta_id += 1

dim_metas = pd.DataFrame(metas)

In [5]:
# ── FACT_SALES ─────────────────────────────────────────────

records = []
sale_id = 1

dates = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')

for date in dates:
    month      = date.month
    year       = date.year
    seasonality = SEASONALITY[month]

    # Inflação: preços 8% maiores em 2023
    price_factor = 1.0 if year == 2022 else 1.08

    for _, seller in dim_seller.iterrows():

        if seller['regiao'] == 'Norte':
            weights = [0.55, 0.30, 0.10, 0.05]
        elif seller['anos_experiencia'] > 5:
            weights = [0.15, 0.35, 0.30, 0.20]
        elif seller['anos_experiencia'] > 2:
            weights = [0.25, 0.40, 0.25, 0.10]
        else:
            weights = [0.40, 0.40, 0.15, 0.05]

        adjusted_zero = max(0.05, weights[0] / seasonality)
        total_rest    = 1 - adjusted_zero
        scale         = total_rest / sum(weights[1:])
        adj_weights   = [adjusted_zero] + [w * scale for w in weights[1:]]

        n_sales = random.choices([0, 1, 2, 3], weights=adj_weights)[0]

        for _ in range(n_sales):
            category     = random.choice(CATEGORIES)
            product_name = random.choice(PRODUCTS[category])
            base_price   = BASE_PRICE[category] * price_factor
            unit_price   = round(base_price * random.uniform(0.90, 1.15), 2)

            # Periféricos com desconto alto — intencional
            if category == 'Periféricos':
                discount_pct = round(random.uniform(0.18, 0.28), 2)
            else:
                discount_pct = round(random.uniform(0.02, 0.12), 2)

            # Quantidade realista por categoria
            quantity = random.randint(1, MAX_QTY[category])
            revenue  = round(unit_price * quantity * (1 - discount_pct), 2)
            cost     = round(unit_price * quantity * COST_RATIO[category], 2)
            profit   = round(revenue - cost, 2)

            records.append({
                'venda_id':       sale_id,
                'data':           date.strftime('%Y-%m-%d'),
                'vendedor_id':    seller['vendedor_id'],
                'regiao':         seller['regiao'],
                'categoria':      category,
                'produto':        product_name,
                'quantidade':     quantity,
                'preco_unitario': unit_price,
                'desconto_pct':   discount_pct,
                'receita':        revenue,
                'custo':          cost,
                'lucro':          profit,
            })
            sale_id += 1

fact_sales = pd.DataFrame(records)


In [6]:
# ── VALIDAÇÃO ──────────────────────────────────────────────

print("=" * 50)
print("✅ Dataset gerado com sucesso!")
print("=" * 50)
print(f"\n📋 fact_sales : {len(fact_sales):>8,} linhas")
print(f"👤 dim_seller : {len(dim_seller):>8,} linhas")
print(f"🎯 dim_metas  : {len(dim_metas):>8,} linhas")

print("\n── Receita por Região ──────────────────────────")
region_summary = (
    fact_sales.groupby('regiao')['receita']
    .sum()
    .sort_values(ascending=False)
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(region_summary.to_string())

print("\n── Margem por Categoria ────────────────────────")
margin_summary = (
    fact_sales.groupby('categoria')
    .apply(lambda x: round((x['lucro'].sum() / x['receita'].sum()) * 100, 1))
    .sort_values()
)
for cat, margin in margin_summary.items():
    print(f"  {cat:<15} {margin}%")

print("\n── Receita 2022 vs 2023 ────────────────────────")
yearly = (
    fact_sales.assign(ano=pd.to_datetime(fact_sales['data']).dt.year)
    .groupby('ano')['receita']
    .sum()
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(yearly.to_string())

✅ Dataset gerado com sucesso!

📋 fact_sales :   17,898 linhas
👤 dim_seller :       22 linhas
🎯 dim_metas  :      120 linhas

── Receita por Região ──────────────────────────
regiao
Sudeste         R$ 12,967,125
Nordeste         R$ 9,296,267
Sul              R$ 8,303,954
Centro-Oeste     R$ 7,043,114
Norte            R$ 4,544,595

── Margem por Categoria ────────────────────────
  Periféricos     -3.9%
  Smartphones     20.5%
  Notebooks       22.6%
  Monitores       24.7%
  Acessórios      30.1%

── Receita 2022 vs 2023 ────────────────────────
ano
2022    R$ 20,656,799
2023    R$ 21,498,255


In [7]:
# ── EXPORTAR ───────────────────────────────────────────────

data_dir = os.path.join(os.getcwd(), '..', 'data')
os.makedirs(data_dir, exist_ok=True)

fact_sales.to_csv(os.path.join(data_dir, 'fact_sales.csv'),   index=False, encoding='utf-8-sig')
dim_seller.to_csv(os.path.join(data_dir, 'dim_seller.csv'),   index=False, encoding='utf-8-sig')
dim_metas.to_csv(os.path.join(data_dir,  'dim_metas.csv'),    index=False, encoding='utf-8-sig')

print(f"\n📁 Arquivos salvos em: {data_dir}")
print("=" * 50)


📁 Arquivos salvos em: C:\Users\allis\Documents\Vortex Projeto\python\..\data


In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

# ── CONFIGURAÇÕES ──────────────────────────────────────────

REGIONS    = ['Sul', 'Sudeste', 'Centro-Oeste', 'Norte', 'Nordeste']
CATEGORIES = ['Notebooks', 'Periféricos', 'Smartphones', 'Acessórios', 'Monitores']

PRODUCTS = {
    'Notebooks':    ['Dell Inspiron', 'Lenovo IdeaPad', 'HP Pavilion', 'Asus VivoBook'],
    'Periféricos':  ['Mouse Logitech', 'Teclado Redragon', 'Headset HyperX', 'Webcam Logitech'],
    'Smartphones':  ['Samsung A54', 'Xiaomi 12', 'iPhone 13', 'Motorola Edge'],
    'Acessórios':   ['Cabo USB-C', 'Carregador 65W', 'Suporte Notebook', 'Hub USB'],
    'Monitores':    ['LG 24"', 'Dell 27"', 'Samsung 32"', 'AOC 24"'],
}

BASE_PRICE = {
    'Notebooks':    2800,
    'Periféricos':  180,
    'Smartphones':  1900,
    'Acessórios':   90,
    'Monitores':    1100,
}

# FIX 1: COST_RATIO aplicado sobre base_price
COST_RATIO = {
    'Notebooks':    0.72,
    'Periféricos':  0.70,
    'Smartphones':  0.74,
    'Acessórios':   0.60,
    'Monitores':    0.68,
}

MAX_QTY = {
    'Notebooks':    2,
    'Periféricos':  10,
    'Smartphones':  3,
    'Acessórios':   20,
    'Monitores':    3,
}

SEASONALITY = {
    1:  1.10,
    2:  0.85,
    3:  0.90,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.05,
    8:  0.95,
    9:  0.95,
    10: 1.00,
    11: 1.35,
    12: 1.20,
}

GOALS_2022 = {
    'Sul':          350000,
    'Sudeste':      550000,
    'Centro-Oeste': 310000,
    'Norte':        220000,
    'Nordeste':     390000,
}

GOAL_SEASONALITY = {
    1:  1.05,
    2:  0.90,
    3:  0.95,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.00,
    8:  0.95,
    9:  1.00,
    10: 1.05,
    11: 1.25,
    12: 1.15,
}

GROWTH_2023 = 1.12

# FIX 4: Descontos em passos inteiros de 1%
DISCOUNT_BY_REGION = {
    'Sul':          (2,  10),
    'Sudeste':      (2,  10),
    'Centro-Oeste': (3,  12),
    'Norte':        (5,  15),
    'Nordeste':     (3,  12),
}

PERIFERICO_DISCOUNT_BY_REGION = {
    'Sul':          (12, 20),
    'Sudeste':      (12, 20),
    'Centro-Oeste': (15, 22),
    'Norte':        (18, 25),
    'Nordeste':     (15, 22),
}

# ── DIM_SELLER ─────────────────────────────────────────────

sellers = []
seller_id = 1

for region in REGIONS:
    n_sellers = 6 if region == 'Sudeste' else 4
    for _ in range(n_sellers):
        seniority = round(random.uniform(0.5, 8.0), 1)
        sellers.append({
            'vendedor_id':       seller_id,
            'nome_vendedor':     fake.name(),
            'regiao':            region,
            'anos_experiencia':  seniority,
            'equipe':            'Comercial',
        })
        seller_id += 1

dim_seller = pd.DataFrame(sellers)

# ── DIM_METAS ──────────────────────────────────────────────

metas = []
meta_id = 1

for mes in pd.date_range(start='2022-01-01', end='2023-12-31', freq='MS'):
    year  = mes.year
    month = mes.month

    for regiao, meta_base in GOALS_2022.items():
        meta_ano   = meta_base if year == 2022 else meta_base * GROWTH_2023
        meta_final = round(meta_ano * GOAL_SEASONALITY[month], 2)

        metas.append({
            'meta_id':     meta_id,
            'regiao':      regiao,
            'mes':         mes.strftime('%Y-%m'),
            'meta_mensal': meta_final,
        })
        meta_id += 1

dim_metas = pd.DataFrame(metas)

# ── FACT_SALES ─────────────────────────────────────────────

records = []
sale_id = 1

dates = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')

for date in dates:
    month        = date.month
    year         = date.year
    seasonality  = SEASONALITY[month]
    price_factor = 1.0 if year == 2022 else 1.08

    for _, seller in dim_seller.iterrows():

        if seller['regiao'] == 'Norte':
            weights = [0.55, 0.30, 0.10, 0.05]
        elif seller['anos_experiencia'] > 5:
            weights = [0.15, 0.35, 0.30, 0.20]
        elif seller['anos_experiencia'] > 2:
            weights = [0.25, 0.40, 0.25, 0.10]
        else:
            weights = [0.40, 0.40, 0.15, 0.05]

        adjusted_zero = max(0.05, weights[0] / seasonality)
        total_rest    = 1 - adjusted_zero
        scale         = total_rest / sum(weights[1:])
        adj_weights   = [adjusted_zero] + [w * scale for w in weights[1:]]

        n_sales = random.choices([0, 1, 2, 3], weights=adj_weights)[0]

        for _ in range(n_sales):
            category   = random.choice(CATEGORIES)
            produto    = random.choice(PRODUCTS[category])
            base_price = BASE_PRICE[category] * price_factor

            # Preço de venda varia — custo permanece fixo no base_price
            unit_price = round(base_price * random.uniform(0.90, 1.15), 2)

            # FIX 4: Desconto em passos inteiros
            regiao = seller['regiao']
            if category == 'Periféricos':
                d_min, d_max = PERIFERICO_DISCOUNT_BY_REGION[regiao]
            else:
                d_min, d_max = DISCOUNT_BY_REGION[regiao]

            discount_pct = random.randint(d_min, d_max) / 100

            quantity = random.randint(1, MAX_QTY[category])
            revenue  = round(unit_price * quantity * (1 - discount_pct), 2)

            # FIX 1: Custo baseado no base_price — não no unit_price
            cost   = round(base_price * quantity * COST_RATIO[category], 2)

            # FIX 2: Salvaguarda contra lucro negativo
            profit = max(round(revenue - cost, 2), 0.01)

            records.append({
                'venda_id':       sale_id,
                'data':           date.strftime('%Y-%m-%d'),
                'vendedor_id':    seller['vendedor_id'],
                'regiao':         seller['regiao'],
                'categoria':      category,
                'produto':        produto,
                'quantidade':     quantity,
                'preco_unitario': unit_price,
                'desconto_pct':   discount_pct,
                'receita':        revenue,
                'custo':          cost,
                'lucro':          profit,
            })
            sale_id += 1

fact_sales = pd.DataFrame(records)

# ── VALIDAÇÃO ──────────────────────────────────────────────

print("=" * 50)
print("✅ Dataset gerado com sucesso!")
print("=" * 50)
print(f"\n📋 fact_sales : {len(fact_sales):>8,} linhas")
print(f"👤 dim_seller : {len(dim_seller):>8,} linhas")
print(f"🎯 dim_metas  : {len(dim_metas):>8,} linhas")

print("\n── Receita por Região ──────────────────────────")
region_summary = (
    fact_sales.groupby('regiao')['receita']
    .sum()
    .sort_values(ascending=False)
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(region_summary.to_string())

# FIX 5: Sem warning de depreciação
print("\n── Margem por Categoria ────────────────────────")
margin_summary = (
    fact_sales.groupby('categoria')[['lucro', 'receita']]
    .sum()
    .assign(margem=lambda x: round(x['lucro'] / x['receita'] * 100, 1))
    ['margem']
    .sort_values()
)
for cat, margin in margin_summary.items():
    print(f"  {cat:<15} {margin}%")

print("\n── Desconto Médio por Região ───────────────────")
discount_summary = (
    fact_sales.groupby('regiao')['desconto_pct']
    .mean()
    .sort_values(ascending=False)
    .apply(lambda x: f"{x*100:.1f}%")
)
print(discount_summary.to_string())

print("\n── Receita 2022 vs 2023 ────────────────────────")
yearly = (
    fact_sales.assign(ano=pd.to_datetime(fact_sales['data']).dt.year)
    .groupby('ano')['receita']
    .sum()
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(yearly.to_string())

# FIX 3: Checagem de qualidade
print("\n── Checagem de Qualidade ───────────────────────")
print(f"  Nulos fact_sales  : {fact_sales.isnull().sum().sum()}")
print(f"  Nulos dim_seller  : {dim_seller.isnull().sum().sum()}")
print(f"  Nulos dim_metas   : {dim_metas.isnull().sum().sum()}")
print(f"  Lucros negativos  : {(fact_sales['lucro'] < 0).sum()}")
print(f"  Receita negativa  : {(fact_sales['receita'] < 0).sum()}")
print(f"  Vendas duplicadas : {fact_sales['venda_id'].duplicated().sum()}")

# ── EXPORTAR ───────────────────────────────────────────────

data_dir = os.path.join(os.getcwd(), '..', 'data')
os.makedirs(data_dir, exist_ok=True)

fact_sales.to_csv(os.path.join(data_dir, 'fact_sales.csv'),  index=False, encoding='utf-8-sig')
dim_seller.to_csv(os.path.join(data_dir, 'dim_seller.csv'),  index=False, encoding='utf-8-sig')
dim_metas.to_csv(os.path.join(data_dir,  'dim_metas.csv'),   index=False, encoding='utf-8-sig')

print(f"\n📁 Arquivos salvos em: {data_dir}")
print("=" * 50)

✅ Dataset gerado com sucesso!

📋 fact_sales :   17,914 linhas
👤 dim_seller :       22 linhas
🎯 dim_metas  :      120 linhas

── Receita por Região ──────────────────────────
regiao
Sudeste         R$ 13,179,322
Nordeste         R$ 9,646,951
Sul              R$ 8,235,588
Centro-Oeste     R$ 6,843,083
Norte            R$ 4,426,843

── Margem por Categoria ────────────────────────
  Periféricos     17.1%
  Smartphones     22.4%
  Notebooks       24.6%
  Monitores       28.5%
  Acessórios      37.2%

── Desconto Médio por Região ───────────────────
regiao
Norte           12.2%
Centro-Oeste     9.8%
Nordeste         9.7%
Sudeste          8.0%
Sul              8.0%

── Receita 2022 vs 2023 ────────────────────────
ano
2022    R$ 20,569,406
2023    R$ 21,762,381

── Checagem de Qualidade ───────────────────────
  Nulos fact_sales  : 0
  Nulos dim_seller  : 0
  Nulos dim_metas   : 0
  Lucros negativos  : 0
  Receita negativa  : 0
  Vendas duplicadas : 0

📁 Arquivos salvos em: C:\Users\allis\Do